# 🎧 AcousticBrainz API Exploration

> **Source Status:** Deprecated / Sunset (2022)  
> **API Behavior:** The live REST API endpoints (`https://acousticbrainz.org/api/v1/...`) return HTTP `404 Not Found` errors because MetaBrainz froze the online lookup servers.

---

### 📝 Notebook Evaluation Notes

1. **Attempted Live Extraction:** The code below directly queries the AcousticBrainz REST API for low-level audio features (`tonal`, `rhythm`, `bpm`, `key`, `scale`) across sample MusicBrainz IDs using the project's `fetch_raw_api_sample` helper.
2. **Expected Execution Result:** Because the live API has been discontinued, `fetch_raw_api_sample` catches the `404` errors and returns `None`. As a result, `df_low_level` renders as an **empty DataFrame**.
3. **Offline Bulk Download Alternative:** To run feature extraction on real data, the raw dataset archives (`.tar.zst` files containing JSON records) must be downloaded locally from the official dump server: `https://data.metabrainz.org/pub/musicbrainz/acousticbrainz/dumps/`
4. **Confirmed Schema Fields (Essentia Documentation):** Official MetaBrainz/Essentia documentation confirms that the following fields exist in the offline JSON payloads and can be helpful:
   - **BPM / Tempo:** `rhythm.bpm` (Float value representing track beats per minute)
   - **Key Signature:** `tonal.key_key` (String value, e.g., `"C"`, `"A#"`, `"G"`)
   - **Musical Scale:** `tonal.key_scale` (String value, `"major"` or `"minor"`)
   - **Danceability Score:** `rhythm.danceable` (Float score representing raw danceability metric)

In [1]:
import pandas as pd
from src.agies.integration.api_helpers import fetch_raw_api_sample


HEADERS = {"User-Agent": "AGIES/0.1 (info@dataravers.space)"}
print("Dependencies loaded successfully.")

Dependencies loaded successfully.


## 1. Discover Payload Keys (Raw Inspection)

Fetch a single raw sample to inspect the top-level schema returned by AcousticBrainz without making assumptions about internal structure.

In [2]:
sample_mbid = "f9a46338-72e2-4752-b883-ef8805f63560"
url_low = f"https://acousticbrainz.org/api/v1/{sample_mbid}/low-level"

# Fetch raw JSON payload using generic helper
raw_data = fetch_raw_api_sample(url=url_low, headers=HEADERS)

if raw_data:
    print("✅ Top-Level Keys in Raw Response:")
    print(list(raw_data.keys()))

ERROR:root:API Request Failed for URL https://acousticbrainz.org/api/v1/f9a46338-72e2-4752-b883-ef8805f63560/low-level: 404 Client Error: NOT FOUND for url: https://acousticbrainz.org/api/v1/f9a46338-72e2-4752-b883-ef8805f63560/low-level


## 2. Low-Level Acoustic Feature Summary

Extract core low-level feature descriptors into a pandas DataFrame for evaluation.

In [4]:
SAMPLE_MBIDS = [
    "f9a46338-72e2-4752-b883-ef8805f63560",
    "6704032d-225c-4b0c-9941-865005937400"
]

extracted_samples = []

for mbid in SAMPLE_MBIDS:
    url_low = f"https://acousticbrainz.org/api/v1/{mbid}/low-level"
    raw_data = fetch_raw_api_sample(url=url_low, headers=HEADERS)
    
    if raw_data:
        tonal = raw_data.get("tonal", {})
        rhythm = raw_data.get("rhythm", {})
        
        extracted_samples.append({
            "mbid": mbid,
            "bpm": rhythm.get("bpm"),
            "key": tonal.get("key_key"),
            "scale": tonal.get("key_scale"),
            "danceable_score": rhythm.get("danceable")
        })

df_low_level = pd.DataFrame(extracted_samples)
display(df_low_level)

ERROR:root:API Request Failed for URL https://acousticbrainz.org/api/v1/f9a46338-72e2-4752-b883-ef8805f63560/low-level: 404 Client Error: NOT FOUND for url: https://acousticbrainz.org/api/v1/f9a46338-72e2-4752-b883-ef8805f63560/low-level
ERROR:root:API Request Failed for URL https://acousticbrainz.org/api/v1/6704032d-225c-4b0c-9941-865005937400/low-level: 404 Client Error: NOT FOUND for url: https://acousticbrainz.org/api/v1/6704032d-225c-4b0c-9941-865005937400/low-level


""
